# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
# Unit of analysis: one row = the daily search-and-site performance of ONE
# content page (content_hash_id), for ONE client (client_hash_id), on ONE
# report_date. It bundles that page-day's GSC metrics (impressions, clicks,
# avg position) and GA4 metrics (pageviews, sessions, users, engaged
# sessions, engagement time, traffic-source split) into a single record.

# Time window: development happens on month=2026-03 (mid-panel). The final
# month, 2026-06 (the `_sample` table), is sealed: it's the natural outcome
# window for any past->future label, so it is never used to design label or
# feature logic -- only reserved as a future test month.

# Both claims are verified with real queries in Section 3, not just assumed
# here.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
# FEATURE  (known at report_date, safe as model input):
#   gsc_impressions, gsc_clicks, gsc_avg_position,
#   ga4_engaged_sessions, sessions_organic
#   (kept to 5 for the feature frame below; sessions_direct/referral/social
#   and ga4_pageviews/sessions/users/total_engagement_sec are same-day and
#   available too, just not used in this pass)

# LABEL  (not a column in this table -- a proxy to be derived later):
#   Growth / Recovery / Stable / Decline, built by comparing a page's
#   performance in a FUTURE window against an earlier one. Needs data past
#   March, so it is not built in this notebook -- only prepared for.

# CONTEXT  (identifies the row; not a predictive signal):
#   report_date, client_hash_id, content_hash_id,
#   client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available

# EXCLUDED  (and why):
#   Rows where gsc_data_available is False, from any GSC-based feature or
#   label calculation. A 0 in gsc_impressions on an unavailable-data row
#   means "nothing recorded," not "zero impressions" -- treating a data gap
#   as a real zero would quietly bias the model toward reading missing data
#   as poor performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
!pip install -q datasets duckdb scikit-learn huggingface_hub

from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import duckdb
import pandas as pd

login(token=userdata.get("HF_TOKEN"))

MONTH = "2026-03"  # mid-panel dev month -- never the sealed 2026-06 sample

ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files={"train": f"fact_content_daily_performance/month={MONTH}/data_0.parquet"},
    split="train",
)
df = ds.to_pandas()

con = duckdb.connect()
con.register("panel", df)

print("Rows loaded for", MONTH, ":", len(df))
df.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Rows loaded for 2026-03 : 9841378


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Query 1 — Grain

Prove that (`client_hash_id`, `content_hash_id`, `report_date`) is unique.
If it isn't, the "one row = one page-day" claim in Section 1 is wrong.

In [6]:
grain_check = con.execute("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n_rows
    FROM panel
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()

print("Duplicate (client, content, date) combinations:", len(grain_check))
grain_check.head()
# Expect 0 rows -> confirms the grain stated in Section 1.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, date) combinations: 0


,client_hash_id,content_hash_id,report_date,n_rows


### Query 2 — Row count & date span

Basic shape of this slice: how many rows, how many distinct clients/pages,
and what date range they actually cover (should sit inside March 2026).

In [7]:
slice_shape = con.execute("""
    SELECT
        COUNT(*) AS n_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_pages
    FROM panel
""").df()

slice_shape


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,first_date,last_date,n_clients,n_pages
0,9841378,2026-03-01,2026-03-31,55,331437


### Query 3 — Missing values / availability, filtered with `IS TRUE`

How many rows survive once I require the source data to actually be
present, instead of counting an unavailable-data zero as a real zero? This
backs the exclusion rule from Section 2.

In [8]:
availability = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM panel
""").df()

availability
# If gsc_available_rows << total_rows, that is real signal loss, not a bug --
# it is why Section 2 excludes gsc_data_available = false rows from GSC
# features.


,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


### Five features

Built from the same March slice, restricted to rows where the relevant
source is actually available. Every feature is known **as of
`report_date`** — none of them look forward.

In [9]:
feat = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_engaged_sessions,
        sessions_organic
    FROM panel
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()

print("Feature frame shape:", feat.shape)
feat.head()

# Available at the decision moment because...
# 1. gsc_impressions      -- logged by Search Console for that exact date;
#                             the day is already over when this row exists.
# 2. gsc_clicks            -- same source, same date, no forward aggregation.
# 3. gsc_avg_position      -- a same-day average of that day\'s positions.
# 4. ga4_engaged_sessions  -- GA4\'s own daily count for report_date.
# 5. sessions_organic      -- same-day traffic-source split from GA4.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (364347, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,sessions_organic
0,client_65de48885f4ef01b,content_5c80451459c29b4a,2026-03-01,5,0,5.400000,0.0,1.0
1,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2026-03-01,39,0,5.666667,0.0,0.0
2,client_65de48885f4ef01b,content_e25ea7297a1dffd3,2026-03-01,179,0,5.156425,0.0,0.0
3,client_65de48885f4ef01b,content_6b0149a80607dac3,2026-03-01,72,0,7.694444,0.0,0.0
4,client_65de48885f4ef01b,content_62673eea26c31c17,2026-03-01,3282,1,6.167885,0.0,1.0


### The trap — deliberate leakage

A **toy proxy label**, confined to March only, exists purely to demonstrate
the mechanic: "did this page clear the March median click count?" This is
*not* the real Growth/Recovery/Stable/Decline label (that needs a window
past March) — it's a stand-in to leak into.

Fit a classifier honestly first, then add one column that's mathematically
derived from the label, watch the score jump toward 1.0, then delete it and
keep the honest number.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# --- Toy proxy label (March-only, this demo only) ---
median_clicks = feat["gsc_clicks"].median()
feat = feat.copy()
feat["toy_label"] = (feat["gsc_clicks"] > median_clicks).astype(int)

honest_features = [
    "gsc_impressions", "gsc_avg_position",
    "ga4_engaged_sessions", "sessions_organic",
]

X, y = feat[honest_features], feat["toy_label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)
honest_f1 = f1_score(y_test, clf.predict(X_test), average="macro")

print(f"Honest macro F1 (no leak): {honest_f1:.3f}")


Honest macro F1 (no leak): 0.814


In [11]:
# --- The trap: add ONE column derived straight from the label ---
# gsc_clicks is literally the column toy_label was thresholded from, so
# putting it back in hands the model the answer.
leaky_features = honest_features + ["gsc_clicks"]

X_leak = feat[leaky_features]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y, test_size=0.3, random_state=42, stratify=y
)

clf_leak = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leak.fit(X_train_l, y_train_l)
leaky_f1 = f1_score(y_test_l, clf_leak.predict(X_test_l), average="macro")

print(f"Leaky macro F1  (gsc_clicks included): {leaky_f1:.3f}")
print(f"Honest macro F1 (gsc_clicks removed):  {honest_f1:.3f}")
print(f"Jump caused by the leak: +{leaky_f1 - honest_f1:.3f}")

# Fix: drop gsc_clicks and keep the HONEST score above as the number that
# means something. Same lesson as notebook 02 -- leakage inflates the
# metric, not the model.


Leaky macro F1  (gsc_clicks included): 1.000
Honest macro F1 (gsc_clicks removed):  0.814
Jump caused by the leak: +0.186


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [12]:
# Named limitation: only 3.7% of March rows have BOTH GSC and GA4 data
# available (364,347 out of 9,841,378 -- see Query 3). GA4 coverage alone is
# just 4.2% of rows. This isn't mainly because clients lack GA4: 43 of 55
# clients have GA4 connected (client_has_ga4 = True), but daily GA4 records
# are still missing for most page-days even among connected clients. So the
# feature frame in Section 3 is built from a small, GA4-heavy slice of the
# month -- not representative of the full 9.8M-row panel -- and any model
# trained on it is learning from clients/pages where GA4 happens to be
# reporting that day, not from typical page behavior.

# Also: this notebook only proves grain, count/span, and availability for
# ONE month (2026-03). A different month could have a different availability
# rate -- these facts are not verified to hold across the whole panel.

# One more limitation, from the leakage trap in Section 3: a macro F1 near
# 1.0 on this kind of data should be treated as a red flag, not a result --
# it took only one label-derived column to fake a "perfect" model here, so
# any future score close to 1.0 needs to be checked for leakage before it's
# trusted.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.